In [11]:
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader, DirectoryLoader

In [12]:
pdf_path = "../data"
persist_path = "faiss"
EMBEDDING_NAME = "intfloat/multilingual-e5-small"

# Mapping từ school_symbol sang school_name
NAME_MAPPING = {
    "aof": "Học viện Tài chính",
    "ftu": "Trường Đại học Ngoại thương",
    "haui": "Đại học Công nghiệp Hà Nội",
    "hcmus": "Trường Đại học Khoa học Tự nhiên - Đại học Quốc gia Thành phố Hồ Chí Minh",
    "hup": "Trường Đại học Dược Hà Nội",
    "hnue": "Trường Đại học Sư phạm Hà Nội",
    "huce": "Trường Đại học Xây dựng Hà Nội",
    "hust": "Đại học Bách khoa Hà Nội",
    "neu": "Đại học Kinh tế Quốc dân",
    "ptit": "Học viện Công nghệ Bưu chính Viễn thông",
    "ueb": "Trường Đại học Kinh tế - Đại học Quốc gia Hà Nội",
    "ued": "Trường Đại học Giáo dục - Đại học Quốc gia Hà Nội",
    "uet": "Trường Đại học Công nghệ - Đại học Quốc gia Hà Nội",
    "ulis": "Trường Đại học Ngoại ngữ - Đại học Quốc gia Hà Nội",
    "ussh": "Trường Đại học Khoa học Xã hội và Nhân văn - Đại học Quốc gia Hà Nội",
    "utc": "Đại học Giao thông vận tải"
}

In [13]:
def load_all():
    # Sử dụng DirectoryLoader để đọc tất cả file PDF
    loader = DirectoryLoader(pdf_path, glob="**/*.pdf", loader_cls=PyPDFLoader, recursive=True)
    docs = loader.load()
    print(f"Đã load {len(docs)} documents từ PDF files")
    
    # Thay thế metadata chỉ với school và school_symbol
    for doc in docs:
        # Lấy đường dẫn file từ metadata gốc
        source_path = doc.metadata.get('source', '')
        
        # Parse school_symbol từ đường dẫn (ví dụ: ../data/uet/file.pdf -> uet)
        path_parts = source_path.replace('\\', '/').split('/')
        school_symbol = None
        for part in path_parts:
            if part in NAME_MAPPING:
                school_symbol = part
                break
        
        if school_symbol:
            school_name = NAME_MAPPING[school_symbol]
            # Thay thế hoàn toàn metadata cũ bằng metadata mới
            doc.metadata = {
                'school': school_name,
                'school_symbol': school_symbol
            }
        else:
            print(f"Không tìm thấy school_symbol cho file: {source_path}")
            # Nếu không tìm thấy school, để metadata rỗng hoặc thông tin mặc định
            doc.metadata = {
                'school': 'Unknown',
                'school_symbol': 'unknown'
            }
    
    # Tạo text splitter và embedding
    embedding = HuggingFaceEmbeddings(model_name=EMBEDDING_NAME)
    splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128, separators=["\n\n", "\n", " ", ""])
    
    # Chia nhỏ documents thành chunks
    chunks = splitter.split_documents(docs)
    print(f"Đã tạo {len(chunks)} chunks")
    
    # Tạo FAISS vector store
    print("Đang tạo vector store...")
    faiss = FAISS.from_documents(chunks, embedding=embedding)
    faiss.save_local(persist_path)
    print("Hoàn thành!")

In [14]:
load_all()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 71 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)
Ignoring wrong pointing object 75 0 (offset 0)
Ignoring wrong pointing object 71 0 (offset 0)
Ignoring wrong p

Đã load 3003 documents từ PDF files
Đã tạo 7178 chunks
Đang tạo vector store...
Đã tạo 7178 chunks
Đang tạo vector store...
Hoàn thành!
Hoàn thành!
